# Session 6: Exercises

**Do these before session 7.** About ninety minutes; the ⭐⭐⭐ ones join the session's ideas together.

Same pattern as before: **predict, then measure.** Write down what you expect before you run a cell.
Predictions and answers go in the cells marked **✍️**; code to write is marked `# TODO`. Each
exercise has a check cell that says `passed` or tells you what's wrong.

Every check runs on a **scripted model** (lesson appendix A), so exercises 1 to 5 need no key and cost
nothing. Cells marked **live** call the real model through OpenRouter; exercise 6 is the only one that
makes more than a handful of calls.

Worked solutions are released after the next session.

## Setup (just run it)

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import AIMessage, HumanMessage, RemoveMessage, ToolMessage
from langchain_core._api import LangChainBetaWarning
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langgraph.checkpoint.memory import InMemorySaver

sys.path.insert(0, str(Path.cwd() / "project"))
from config import settings                                          # noqa: E402
from agent import build_agent, new_ticket_id, run_config, transcript  # noqa: E402
from tools import TOOLS                                              # noqa: E402
from triage import PROMPTS                                           # noqa: E402
from utils import build_model, make_model                            # noqa: E402

logging.getLogger("google_genai.types").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=LangChainBetaWarning)     # lesson section 8.4


class ScriptedModel(GenericFakeChatModel):
    """Replays the messages it's given and remembers what it was shown on each call."""

    seen: list = []
    disable_streaming: bool = True      # a reply that is only a tool call has no text to stream

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, *args, **kwargs):
        self.seen.append([m.text for m in messages])
        return super()._generate(messages, *args, **kwargs)


def lookup(name: str, args: dict, call_id: str) -> AIMessage:
    """A scripted model turn that asks for one tool."""
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}])


def scripted_agent(replies, *middleware):
    """A real create_agent with real tools and a checkpointer, on a scripted model."""
    model = ScriptedModel(messages=iter(replies), seen=[])
    return create_agent(model, TOOLS, middleware=list(middleware), checkpointer=InMemorySaver()), model


print(f"{settings.APP_NAME} v{settings.VERSION} · {settings.model_id} · key {'found' if settings.has_key else 'missing'}")

## ⭐ Exercise 1: Ticket reality

**The idea.** Lesson sections 2 and 4. Memory is a checkpointer plus the same thread id on every call,
and the thread id is the only thing between two customers' tickets.

Write `turn(agent, text, ticket)`: send **only** the new message, on the given ticket, and return the
final state. Then the check runs two turns on ticket A and the follow-up on ticket B, and inspects
what the scripted model was shown each time.

**✍️ Predict first:** on its third call (ticket B), how many messages does the model receive?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def turn(agent, text: str, ticket: str) -> dict:
    # TODO: invoke the agent with a one-message list and the config for this ticket
    #       (agent.py has a helper that builds it). Return the result.
    return {}

In [ ]:
def check_1():
    agent, model = scripted_agent([AIMessage("Noted."), AIMessage("Yes, ORD-5003."), AIMessage("Which order?")])
    a, b = new_ticket_id(), new_ticket_id()
    turn(agent, "My card ends 4242. Where is ORD-5003?", a)
    turn(agent, "Is that the smartwatch one?", a)
    turn(agent, "Is that the smartwatch one?", b)
    if len(model.seen) != 3:
        return f"expected 3 model calls, saw {len(model.seen)}: is turn() invoking the agent?"
    if "My card ends 4242. Where is ORD-5003?" not in model.seen[1]:
        return "ticket A's second call didn't see its first message: is the thread id in the config?"
    if model.seen[2] != ["Is that the smartwatch one?"]:
        return f"ticket B was shown {model.seen[2]}: something leaked across tickets"
    return "passed"


print(check_1())

**Live (optional, about four small calls).** The same three turns on the real agent:

In [ ]:
v3 = build_agent(InMemorySaver())
a, b = new_ticket_id(), new_ticket_id()
turn(v3, "I'm tanvir@example.com. Where is my order ORD-5003?", a)
print(transcript({"messages": turn(v3, "And is that the one with the smartwatch in it?", a)["messages"][-2:]}))
print()
print(transcript(turn(v3, "And is that the one with the smartwatch in it?", b)))

**✍️** In two sentences: why is ticket B a stranger, and why is that a feature for a support desk
rather than a bug? (Think about the card number in ticket A.)

**✍️ Your answer:**

*(write it here)*

## ⭐ Exercise 2: Count the checkpoints

**The idea.** Lesson section 3.1. A checkpoint is written after every step, not every turn, and v3's
summarization check is a step of its own. That's why `MAX_STEPS` went from 12 to 18.

Write `predicted_checkpoints(lookups, with_check)`: how many checkpoints does **one customer turn** on
a fresh ticket write, if the model makes `lookups` tool calls one at a time before answering, with or
without a middleware check before each model call? Work it out from the step list in section 3.1;
don't measure first.

**✍️ Predict first:** your formula, in words.

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def predicted_checkpoints(lookups: int, with_check: bool) -> int:
    # TODO: one checkpoint for the input, then one per step of the loop.
    return 0

In [ ]:
def measured_checkpoints(lookups: int, with_check: bool) -> int:
    replies = [lookup("get_order", {"order_id": "ORD-5003"}, f"call_{n}") for n in range(lookups)]
    replies.append(AIMessage("Done."))
    middleware = [SummarizationMiddleware(model=ScriptedModel(messages=iter([])), trigger=("tokens", 10**6))]
    agent, _ = scripted_agent(replies, *(middleware if with_check else []))
    config = run_config(new_ticket_id())
    agent.invoke({"messages": [{"role": "user", "content": "Where is ORD-5003?"}]}, config)
    return len(list(agent.get_state_history(config)))


def check_2():
    for with_check in (False, True):
        for lookups in (0, 1, 2, 3):
            want, got = measured_checkpoints(lookups, with_check), predicted_checkpoints(lookups, with_check)
            if want != got:
                return f"lookups={lookups}, with_check={with_check}: measured {want}, you predicted {got}"
    return "passed"


print(check_2())

**✍️** `recursion_limit` counts steps, not checkpoints (the input isn't a step). With the check, what
is the smallest limit that allows five lookups and the answer? And what happens to it when session 7
adds three more middleware hooks before each model call?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐ Exercise 3: The debugging view

**The idea.** Lesson section 8.2. `stream.values` yields the whole state after each step, so you can
see the exact moment something lands.

Write `first_result(stream)`: read `stream.values` and return `(index, tool_name)` for the **first**
snapshot whose last message is a tool result. Return `(None, None)` if no tool ran.

**✍️ Predict first:** for a run that calls `lookup_customer` then `get_shipment`, which snapshot
index holds the first tool result? (Snapshot 0 is the customer's message.)

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def first_result(stream) -> tuple[int | None, str | None]:
    # TODO: enumerate(stream.values); each item is a state dict with "messages".
    #       Remember the stream is lazy, and finish it before returning (lesson section 8.3).
    return None, None

In [ ]:
def check_3():
    script = [lookup("lookup_customer", {"email": "tanvir@example.com"}, "c1"),
              lookup("get_shipment", {"order_id": "ORD-5003"}, "c2"), AIMessage("On its way.")]
    agent, _ = scripted_agent(script)
    config = run_config(new_ticket_id())
    stream = agent.stream_events({"messages": [{"role": "user", "content": "Where is my parcel?"}]},
                                 config, version="v3")
    index, name = first_result(stream)
    if name != "lookup_customer":
        return f"expected the first result to come from lookup_customer, got {name!r}"
    if index != 2:
        return f"expected snapshot 2, got {index}"
    if len(agent.get_state(config).values["messages"]) != 6:
        return "the run didn't finish: did you drive the stream to the end?"
    return "passed"


print(check_3())

**Live (optional, about three calls).** Run it on the real agent with the lesson's question:

In [ ]:
stream = v3.stream_events({"messages": [{"role": "user", "content":
                           "I'm tanvir@example.com. Where is my order ORD-5003?"}]},
                          run_config(new_ticket_id()), version="v3")
print(first_result(stream))

**✍️** A customer says the bot told them the wrong delivery date. Which view do you open first,
`stream.values` (or its LangSmith equivalent) or the final reply, and what are you looking for?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐ Exercise 4: A safe cut

**The idea.** Lesson section 10.1. A list slice can keep a tool result whose request was cut off, or
open with the model's own tool call. Some providers reject that, others quietly accept a fragment.

Write `safe_cut(messages, keep)`: return at most the last `keep` messages, moving the cut **later**
until the kept part starts with a customer message. If no customer message fits, return `[]`.

**✍️ Predict first:** on the ticket below, is any `keep` value safe to slice naively? Which?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def sample_ticket() -> list:
    """Three turns: a lookup, a question with two lookups, a plain answer."""
    return [
        HumanMessage("Where is ORD-5003?", id="h1"),
        lookup("get_shipment", {"order_id": "ORD-5003"}, "t1"),
        ToolMessage('{"status": "in_transit"}', tool_call_id="t1", id="r1"),
        AIMessage("In transit.", id="a1"),
        HumanMessage("And the kettle and the watch?", id="h2"),
        lookup("get_product", {"sku_or_name": "kettle"}, "t2"),
        ToolMessage('{"stock": 0}', tool_call_id="t2", id="r2"),
        lookup("get_product", {"sku_or_name": "watch"}, "t3"),
        ToolMessage('{"stock": 12}', tool_call_id="t3", id="r3"),
        AIMessage("Kettle out of stock, watch in stock.", id="a2"),
        HumanMessage("Thanks!", id="h3"),
        AIMessage("You're welcome.", id="a3"),
    ]


def safe_cut(messages: list, keep: int) -> list:
    # TODO
    return messages[-keep:]

In [ ]:
def problems(kept: list, original: list, keep: int) -> str | None:
    if len(kept) > keep:
        return f"kept {len(kept)} messages, more than {keep}"
    if kept and kept != original[len(original) - len(kept):]:
        return "the result isn't the tail of the ticket"
    if kept and kept[0].type != "human":
        return f"starts with a {kept[0].type} message"
    asked = set()
    for m in kept:
        asked |= {c["id"] for c in getattr(m, "tool_calls", [])}
        if m.type == "tool" and m.tool_call_id not in asked:
            return f"result {m.tool_call_id} kept without its request"
    return None


def check_4():
    ticket = sample_ticket()
    for keep in range(1, len(ticket) + 1):
        problem = problems(safe_cut(ticket, keep), ticket, keep)
        if problem:
            return f"keep={keep}: {problem}"
    return "passed"


print(check_4())

**✍️** `trim_messages(..., strategy="last", start_on="human")` does the same job by tokens. Why
might a production system still prefer to cut by messages, or by turns? And why is "start on a
customer message" enough to guarantee every kept result has its request?

**✍️ Your answer:**

*(write it here)*

## ⭐⭐⭐ Exercise 5: A handoff summary

**The idea.** Lesson sections 3 and 10.4. A human agent taking over a ticket wants a summary. There
are two ways to get one, and they differ in what they leave behind on the ticket.

Write both:

- `summary_as_turn(agent, ticket)`: send *"Summarize this ticket so far for a human colleague"* as a
  normal turn on the ticket and return the reply text.
- `summary_read_only(agent, model, ticket)`: read the ticket's messages with `get_state`, call `model`
  directly with a system prompt plus those messages, and return the reply text **without writing
  anything to the ticket**.

**✍️ Predict first:** after each one, how many messages does the ticket hold?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
HANDOFF = "Summarize this ticket so far for a human colleague: who, what was asked, what was found, what's open."


def summary_as_turn(agent, ticket: str) -> str:
    # TODO
    return ""


def summary_read_only(agent, model, ticket: str) -> str:
    # TODO: messages = agent.get_state(run_config(ticket)).values["messages"]
    return ""

In [ ]:
def check_5():
    agent, _ = scripted_agent([AIMessage("In transit."), AIMessage("SUMMARY A")])
    reader = ScriptedModel(messages=iter([AIMessage("SUMMARY B")]), seen=[])
    ticket = new_ticket_id()
    agent.invoke({"messages": [{"role": "user", "content": "Where is ORD-5003?"}]}, run_config(ticket))

    count = lambda: len(agent.get_state(run_config(ticket)).values["messages"])
    before = count()
    if summary_read_only(agent, reader, ticket) != "SUMMARY B":
        return "summary_read_only didn't return the model's reply"
    if count() != before:
        return f"summary_read_only changed the ticket: {before} -> {count()} messages"
    if "Where is ORD-5003?" not in reader.seen[0]:
        return "the read-only model wasn't shown the ticket"
    if summary_as_turn(agent, ticket) != "SUMMARY A":
        return "summary_as_turn didn't return the agent's reply"
    if count() != before + 2:
        return f"summary_as_turn should add a question and an answer: {before} -> {count()}"
    return "passed"


print(check_5())

Now add the read-only version to `chat.py` as a `/summary` command. Copy `project/` somewhere of your
own first: the snapshot stays as the session left it. Two lines in `main()`'s command chain, and
`summary_read_only` imported or pasted in. Try it on a three-turn conversation.

**✍️** Which version belongs behind a "hand over to a human" button, and why? Think about who reads
the ticket next, what it costs on every later turn, and section 4's privacy boundary.

**✍️ Your answer:**

*(write it here)*

## ⭐⭐⭐ Exercise 6 [ADV]: A cost meter

**The idea.** Lesson sections 9 and 10. Watch the bill per turn, with and without a bound.

Write `cost_meter(agent, turns)`: run the turns on one fresh ticket and return a list with one entry
per turn: the **cumulative** input tokens spent so far, every model call included (the summarizer's
too). `get_usage_metadata_callback()` from the lesson's section 10.4 does the counting.

**✍️ Predict first:** over six turns, roughly how much less will the summarized agent spend, if
anything? When does summarization start paying for its own extra call?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def cost_meter(agent, turns: list[str]) -> list[int]:
    # TODO
    return []

In [ ]:
def check_6():
    agent, _ = scripted_agent([AIMessage("one"), AIMessage("two"), AIMessage("three")])
    totals = cost_meter(agent, ["a", "b", "c"])
    if len(totals) != 3:
        return f"expected one total per turn, got {totals}"
    if totals != sorted(totals):
        return f"cumulative totals can't go down: {totals}"
    return "passed (the scripted model reports no tokens, so the live run below is the real test)"


print(check_6())

**Live (about 30 small calls, a few minutes with the rate limiter).**

In [ ]:
TURNS = [
    "Hi, I'm tanvir@example.com. Where is my order ORD-5003?",
    "If I'm out, please ask the courier to leave it with the guard at gate 2.",
    "Call me on 01711-000222 rather than emailing, any time after 6pm.",
    "Is the BrewMaster kettle in stock?",
    "What warranty does the PulseFit smartwatch have?",
    "How much did I pay for the watch?",
]
plain = create_agent(build_model(), TOOLS, system_prompt=PROMPTS["agent_prompt"], checkpointer=InMemorySaver())
bounded = create_agent(build_model(), TOOLS, system_prompt=PROMPTS["agent_prompt"], checkpointer=InMemorySaver(),
                       middleware=[SummarizationMiddleware(model=make_model(settings.model_id),
                                                           trigger=("tokens", 600), keep=("messages", 6),
                                                           summary_prompt=PROMPTS["summary_prompt"])])
for label, agent in [("no bound", plain), ("summarized", bounded)]:
    print(f"{label:11}", cost_meter(agent, TURNS))

**✍️** Where do the two lines cross, if they do? Name one thing summarization costs that this meter
doesn't show, and one thing it saves that the meter doesn't show either.

**✍️ Your answer:**

*(write it here)*